In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# ─── paths & constants ────────────────────────────────────────────────────
GND_PATH   = "./3D-Data/Measured/gnd_n=50_noise=0.0.csv"
MODEL_DIR  = Path("./predictions/gps_log_d")       # log-d GP folder
OUT_DIR    = Path("./gp_logd_noise_stats")         # ← summary CSVs go here
OUT_DIR.mkdir(exist_ok=True)

NOISE_LEVELS = [0.0, 6e-06, 1.9e-05, 6e-05, 1.9e-04]
FILE_PATTERN = "rss_n={n}_noise={noise}_seed={seed}.csv"

# ─── helpers ──────────────────────────────────────────────────────────────
def compute_difference_norms(df1, df2):
    diff = df1[["X", "Y", "Z"]] - df2[["X", "Y", "Z"]]
    return np.linalg.norm(diff, axis=1)

df_gnd = pd.read_csv(GND_PATH)[["X", "Y", "Z"]]

def load_errors(n_train: int, noise: float) -> np.ndarray:
    seeds = range(10) if noise == 0.0 else [0]
    errs  = []
    for s in seeds:
        f = MODEL_DIR / FILE_PATTERN.format(n=n_train, noise=noise, seed=s)
        errs.append(compute_difference_norms(df_gnd, pd.read_csv(f)))
    return np.concatenate(errs)

# ─── loop over n and save CSVs ────────────────────────────────────────────
for n_train in range(50, 1501, 50):

    rows = []
    for nz in NOISE_LEVELS:
        err = load_errors(n_train, nz)
        rows.append({
            "noise":        nz,
            "p05":  np.percentile(err, 5),
            "p25":  np.percentile(err, 25),
            "p50":  np.percentile(err, 50),
            "p75":  np.percentile(err, 75),
            "p95":  np.percentile(err, 95),
        })

    df_stats = pd.DataFrame(rows)
    out_file = OUT_DIR / f"gp_logd_noise_stats_n_{n_train}.csv"
    df_stats.to_csv(out_file, index=False)

print("✓ All summary files written to", OUT_DIR.resolve())


✓ All summary files written to /home/sumo/anikraft/RSS/chapter4_recreating_results/gp_logd_noise_stats


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# ─── paths & metadata ────────────────────────────────────────────────────
GND_PATH  = "./3D-Data/Measured/gnd_n=50_noise=0.0.csv"    # 125 000-row ground truth
PRED_ROOT = Path("./predictions")                           # folder with model subdirs
OUT_DIR   = Path("./axis_error_percentiles")                # summary CSVs will go here
OUT_DIR.mkdir(exist_ok=True)

SEEDS = range(10)                                           # 0-9 for every n
AXES  = ["X", "Y", "Z"]

REPRESENTATIONS = ["absolute", "relative", "log_d"]
REP_NAME        = {"absolute": "Absolute", "relative": "Relative", "log_d": "logD"}

MODELS          = ["gps", "nns", "KANs"]
MODEL_NAME      = {"gps": "GPs", "nns": "NNs", "KANs": "KANs"}

PATTERNS = {
    ("gps",  "absolute"): "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps",  "relative"): "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps",  "log_d"):    "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("nns",  "absolute"): "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns",  "relative"): "relative_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns",  "log_d"):    "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("KANs", "absolute"): "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs", "relative"): "KAN_predictions_relative_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs", "log_d"):    "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv",
}

# read full ground-truth once (shape = 125 000 × 3)
GT = pd.read_csv(GND_PATH)[AXES].to_numpy()

def abs_errors(model, rep, axis_idx, n_train):
    """Return |Δaxis| for all rows × all seeds (concatenated)."""
    errs = []
    base = PRED_ROOT / f"{model}_{rep}"
    for s in SEEDS:
        fn = base / PATTERNS[(model, rep)].format(n=n_train, seed=s)
        pred = pd.read_csv(fn).iloc[:, axis_idx].to_numpy()
        errs.append(np.abs(GT[:, axis_idx] - pred))
    return np.concatenate(errs)         # length = (#rows × #seeds)

# ─── loop over n and write CSVs ───────────────────────────────────────────
for n in range(50, 1501, 50):
    rows = []
    for rep in REPRESENTATIONS:
        for mod in MODELS:
            for idx, ax in enumerate(AXES):
                e = abs_errors(mod, rep, idx, n)
                rows.append({
                    "model":           MODEL_NAME[mod],
                    "representation":  REP_NAME[rep],
                    "axis":            ax,
                    "p05_error_m":     np.percentile(e, 5),
                    "p25_error_m":     np.percentile(e, 25),
                    "p50_error_m":     np.percentile(e, 50),
                    "p75_error_m":     np.percentile(e, 75),
                    "p95_error_m":     np.percentile(e, 95),
                })

    pd.DataFrame(rows).to_csv(
        OUT_DIR / f"axis_error_percentiles_n_{n}.csv", index=False
    )

print("✓ Percentile summaries saved in", OUT_DIR.resolve())


✓ Percentile summaries saved in /home/sumo/anikraft/RSS/chapter4_recreating_results/axis_error_percentiles


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# ─── paths & meta-data (identical to the plotting cell) ────────────────────
GT_PATH = "./3D-Data/Measured/gnd_n=50_noise=0.0.csv"          # ground truth
PRED_ROOT = Path("./predictions")                               # prediction dirs
OUT_DIR   = Path("./total_euclidean_error_stats")               # summary CSVs
OUT_DIR.mkdir(exist_ok=True)

SEEDS = range(10)                                               # 0-9 for every n
REPRESENTATIONS = ["absolute", "relative", "log_d"]
REP_NAME        = {"absolute": "Absolute", "relative": "Relative", "log_d": "logD"}
MODELS          = ["gps", "nns", "KANs"]
MODEL_NAME      = {"gps": "GPs", "nns": "NNs", "KANs": "KANs"}

PATTERNS = {
    ("gps",  "absolute"): "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps",  "relative"): "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("gps",  "log_d"):    "rss_n={n}_noise=0.0_seed={seed}.csv",
    ("nns",  "absolute"): "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns",  "relative"): "relative_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("nns",  "log_d"):    "full_model_rss_n={n}_noise=0.0_seed={seed}_test_noise=0.0.csv",
    ("KANs", "absolute"): "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs", "relative"): "KAN_predictions_relative_rss_n={n}_noise=0.0_seed={seed}.csv",
    ("KANs", "log_d"):    "KAN_predictions_rss_n={n}_noise=0.0_seed={seed}.csv",
}

# ─── helpers ───────────────────────────────────────────────────────────────
GT = pd.read_csv(GT_PATH)[["X", "Y", "Z"]].to_numpy()           # (125 000, 3)

def euclidean_errors(model, rep, n_train):
    """Return concatenated 3-D Euclidean errors for all rows × all seeds."""
    errs = []
    base = PRED_ROOT / f"{model}_{rep}"
    for s in SEEDS:
        fn = base / PATTERNS[(model, rep)].format(n=n_train, seed=s)
        pred = pd.read_csv(fn)[["X", "Y", "Z"]].to_numpy()
        errs.append(np.linalg.norm(GT - pred, axis=1))
    return np.concatenate(errs)                                  # 1.25 M values

# ─── loop over n and write CSVs ────────────────────────────────────────────
for n in range(50, 1501, 50):
    rows = []
    for rep in REPRESENTATIONS:
        for mod in MODELS:
            e = euclidean_errors(mod, rep, n)
            rows.append({
                "model":            MODEL_NAME[mod],
                "representation":   REP_NAME[rep],
                "p05_error_m":      np.percentile(e, 5),
                "p25_error_m":      np.percentile(e, 25),
                "p50_error_m":      np.percentile(e, 50),
                "p75_error_m":      np.percentile(e, 75),
                "p95_error_m":      np.percentile(e, 95),
                "mean_error_m":     e.mean()
            })

    out_file = OUT_DIR / f"total_euclidean_error_stats_n_{n}.csv"
    pd.DataFrame(rows).to_csv(out_file, index=False)

print("✓ All percentiles + mean saved in:", OUT_DIR.resolve())


✓ All percentiles + mean saved in: /home/sumo/anikraft/RSS/chapter4_recreating_results/total_euclidean_error_stats
